# Retrieve singe neuron predictions vs targets 

## Before the experiment: get everything ready 

### imports

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize
from IPython.display import display
import sys
import os
from pathlib import Path


### SET THIS PATH FROM THE CURRENT WORKING DIRECTORY TO THE REPO DIRECTORY
relative_repo_path = "GitRepos/simulation_closed_loop"

In [4]:



# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
full_repo_path = os.path.join(cwd, relative_repo_path)

if not os.path.exists(full_repo_path):
    raise ValueError(f"The specified relative path to the repo does not exist: {full_repo_path}\
                     Check workding dir and relative repo dir path variable.")

print(f"Working directoty: {cwd}")
print(f"Full repo path: {full_repo_path}")

# append repo path 
sys.path.append(full_repo_path)



# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"model_in_the_loop/config")
print(f"Config path relative to cwd: {config_path}")
if not os.path.exists(os.path.join(cwd,config_path)):
    raise ValueError(f"The specified config path does not exist: {os.path.join(cwd,config_path)}\
                     Check workding dir and config dir path variable.")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")



Working directoty: /gpfs01/euler/User/ssuhai
Full repo path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop
Config path relative to cwd: GitRepos/simulation_closed_loop/model_in_the_loop/config
Configuration loaded successfully:


In [5]:
from model_in_the_loop.core import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper,
                                                     )

from model_in_the_loop.utils.plotting import show_all_rois_plot
from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure

from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata
from model_in_the_loop.utils.simple_logging import log

### Create processing components (connect them to DB)

In [6]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [7]:
# # # Load config and tables
dj_table_holder.setup()

[2025-09-10 10:26:10,649][INFO]: Connecting ssuhai@172.25.240.200:3306


[2025-09-10 10:26:10,693][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/core/dj_wrappers/base.py:288: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [8]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [28]:
dict(cfg.ini_file_keys)

{'string_userName': 'clexperimenter', 'string_projName': 'projectname'}

In [30]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)


copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
    ini_file_static_args=cfg.ini_file_keys, # type: ignore
)

SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1/Raw/M1_RR_GCL4_DN_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1/Raw/M1_RR_GCL4_MC18_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1/Raw/M1_RR_GCL4_DN_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1/Raw/M1_RR_GCL4_chirp_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1/Raw/M1_RR_GCL4_MB_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/2025

### Preprocessing

In [31]:
preprocessor.upload_iteration_metadata()

/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/experiment.py:154: UserWarning: Eye is set to right in .ini file, but exp_num is 1 and header_file_name is '20250909_right.ini'. Use exp_num=1 for left eye and exp_num=2 for right eye. To overwrite this, use all-caps in .ini file which is then used.
  warnings.warn(


Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop//model_in_the_loop/data/data_dj_format/20250909/1
		header_name: 20250909_right.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 9, 9, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 24.68it/s]

Found 4 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1}
	Adding field: `{'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1}`



Processes: 100%|██████████| 6/6 [00:08<00:00,  1.46s/it]


In [33]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

Missing field key found: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0'}


In [35]:
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()


field_key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0'} 
pres_keys: [{'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0', 'stim_name': 'densenoise', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0', 'stim_name': 'gChirp', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0', 'stim_name': 'mouse_cam', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 9, 9), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL4', 'region': 'RR', 'cond1': 'iter0', 'stim_name': 'movingbar', 'cond2': 'control'}]
No ROI masks found for field: {'experimenter': 'c

Returned InteractiveRoiCanvas object. To start GUI, call <enter_object_name>.start_gui().


Processes: 100%|██████████| 404/404 [00:01<00:00, 205.40it/s]


### Visualize cell tpe and quality

In [36]:
quality_type_analysis_wrapper.compute_analysis(field_key=field_key)


Processes: 100%|██████████| 101/101 [00:00<00:00, 196.90it/s]
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/classifier/rgc_classifier.py:402: UserWarning: Parallel processing not implemented!
  warnings.warn('Parallel processing not implemented!')
CelltypeAssignment: 100%|██████████| 1/1 [00:06<00:00,  6.33s/it]


In [43]:
# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence=cfg.quality_filtering["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"{len(passed_roi_ids_chirp_mb)} ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")


Number filtered rois based on classifier confidence 67.
Number filtered rois based on celltypes 77.
Number filtered rois based on Chirp MB QI 67.
Found 11 rois passing the criterion out of 101 rois.                (d_qi_min=0.6, chrip qidx_min=0.35, celltypes=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32], classifier_confidence=0.25)
11 ROIs passed quality chirp mb filtering: [18, 27, 36, 39, 57, 63, 66, 72, 75, 81, 96]


In [ ]:
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


Processes: 100%|██████████| 11/11 [00:00<00:00, 15.78it/s]


In [45]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min= 0.3)#cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"{len(passed_roi_ids_sta)} ROIs passed STA filtering with rf_qidx_min={cfg.quality_filtering["rf_qidx_min"]}: {passed_roi_ids_sta}")


7 ROIs passed STA filtering with rf_qidx_min=0.5: [18, 27, 39, 57, 63, 66, 75]


In [ ]:
# compute
random_seed_mei_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_sta,
    )




### save data

In [11]:
from model_in_the_loop.utils.model_training import get_single_neuron_session_correlations
from openretina.data_io.hoefling_2024.dataloaders import natmov_dataloaders_v2
from model_in_the_loop.utils.model_training import load_stimuli


In [12]:
data_loader_kwargs = {
"batch_size": 64,
"train_chunk_size": 50,
"allow_over_boundaries": True,
"validation_clip_indices": [0, 4, 10, 11, 18, 30, 45, 62, 67, 77, 79, 80, 81, 83, 95],

}

movies_dict = load_stimuli(cfg.model_configs)

dataloader = natmov_dataloaders_v2(
        neuron_data_dictionary=random_seed_mei_wrapper.neuron_data_dict,
        movies_dictionary=movies_dict,
        **data_loader_kwargs
    )

AttributeError: 'RandomSeedMEIWrapper' object has no attribute 'neuron_data_dict'

In [ ]:
test_corrs, test_pred_targets = get_single_neuron_session_correlations(dataloader, random_seed_mei_wrapper.model,split="test",return_traces=True)

In [78]:
import pickle

path = "/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/analysis_closed_loop_experiments/model_performance_on_online_data"

with open(os.path.join(path,"test_corrs.pkl"),"wb") as f:
    pickle.dump(test_corrs,f)

with open(os.path.join(path,"test_pred_targets.pkl"),"wb") as f:
    pickle.dump(test_pred_targets,f)

In [92]:
train_corrs, train_pred_targets = get_single_neuron_session_correlations(dataloader, random_seed_mei_wrapper.model,split="train",return_traces=True)
with open(os.path.join(path,"train_corrs.pkl"),"wb") as f:
    pickle.dump(train_corrs,f)

with open(os.path.join(path,"train_pred_targets.pkl"),"wb") as f:
    pickle.dump(train_pred_targets,f)



val_corrs, val_pred_targets = get_single_neuron_session_correlations(dataloader, random_seed_mei_wrapper.model,split="validation",return_traces=True)
with open(os.path.join(path,"val_corrs.pkl"),"wb") as f:
    pickle.dump(val_corrs,f)

with open(os.path.join(path,"val_pred_targets.pkl"),"wb") as f:
    pickle.dump(val_pred_targets,f)

In [83]:
from thesis.code.analysis_closed_loop_experiments.model_performance_online_data.utils_model_performance_on_online_data import plot_single_neuron_predicted_actual

### Get comparison data

In [ ]:
# get standard openretina neuron_data_dict 

from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses
from openretina.utils.file_utils import get_local_file_path
from openretina.utils.h5_handling import load_h5_into_dict


In [101]:
cfg.model_configs.paths

{'load_model_path': '${paths.repo_directory}model_in_the_loop/models/full_ensemble', 'cache_dir': '${oc.env:OPENRETINA_CACHE_DIRECTORY}', 'log_dir': '${paths.repo_directory}model_in_the_loop', 'output_dir': 'output', 'movies_path': 'https://huggingface.co/datasets/open-retina/open-retina/blob/main/euler_lab/hoefling_2024/stimuli/rgc_natstim_18x16_joint_normalized_2024-01-11.zip', 'responses_path': 'https://huggingface.co/datasets/open-retina/open-retina/resolve/main/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip'}

In [16]:
# construct a data loader from this 
responses_path = "/gpfs01/euler/User/ssuhai/openretina_cache/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5"

responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.model_configs.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

Original dataset contains 7863 neurons over 67 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (67 remaining)
Overall, dropped 3034 neurons of non-target cell types (-38.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (67 remaining)
Overall, dropped 980 neurons over quality checks (-20.29%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 705 neurons with classifier confidences below 0.25 (-18.32%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 3144 neurons over 67 fields
Total number of cells dropped: 4719 (-60.02%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/67 [00:00<?, ?it/s]

In [17]:
test_session = 'session_1_ventral1_20200226'
neuron_data_dict_or = {test_session: final_responses[test_session]}


dataloader_or = natmov_dataloaders_v2(
        neuron_data_dictionary=neuron_data_dict_or,
        movies_dictionary=movies_dict,
        **data_loader_kwargs
    )

Creating movie dataloaders:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
test_corrs_or, test_pred_targets_or = get_single_neuron_session_correlations(dataloader_or, random_seed_mei_wrapper.model,split="test",return_traces=True)
val_corrs_or, val_pred_targets_or = get_single_neuron_session_correlations(dataloader_or, random_seed_mei_wrapper.model,split="validation",return_traces=True)
train_corrs_or, train_pred_targets_or = get_single_neuron_session_correlations(dataloader_or, random_seed_mei_wrapper.model,split="train",return_traces=True)

In [111]:
import numpy as np  

In [129]:
# save the comparison openretina data to the same path 
with open(os.path.join(path,f"test_corrs_comparison_session{test_session}.pkl"),"wb") as f:
    pickle.dump(test_corrs_or,f)

with open(os.path.join(path,f"val_corrs_comparison_session{test_session}.pkl"),"wb") as f:
    pickle.dump(val_corrs_or,f)

with open(os.path.join(path,f"train_corrs_comparison_session{test_session}.pkl"),"wb") as f:
    pickle.dump(train_corrs_or,f)